# FIRELINE — Audit dataset & batas wilayah

Notebook 05 adalah pintu masuk paket peta prioritas. Kita menjawab **data ini sebenarnya mengamati apa, kapan, dan di mana?** Semua angka dihitung dari berkas lokal yang tercantum dalam konfigurasi. Dataset utama tetap NASA FIRMS VIIRS NOAA-20.

Ini analisis historis untuk fitur 02 keputusan PMA/UX. Riwayat notebook Maria 01–04 dipertahankan untuk referensi; angka resminya untuk paket ini berasal dari notebook 05–06.

Urutan baca: arti satu baris → audit kualitas → batas wilayah → cakupan waktu → makna untuk produk.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
ROOT = Path.cwd() if (Path.cwd() / "fireline").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from fireline.pipeline import settings, audit, build
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
config = settings(ROOT / "config.local.json")

## 1. Arti satu baris

`latitude` (lintang) seperti posisi atas–bawah pada globe: utara atau selatan khatulistiwa. `longitude` (bujur) adalah posisi timur–barat. Keduanya bersama-sama menentukan posisi, bukan skor bahaya.

Satu baris adalah observasi anomali panas satelit. Satu kejadian dapat menghasilkan banyak baris, dari beberapa piksel atau lintasan. FRP dalam MW mengukur daya radiasi panas; `scan × track` adalah footprint piksel pengamatan dan tidak kita ubah menjadi luas terbakar.

`acq_date` dan `acq_time` diparsing sebagai UTC (HHMM, termasuk nol di depan). Dashboard menampilkan waktu observasi lokal WIB/WITA sesuai provinsi; pengelompokan hari tetap UTC agar batas hari konsisten.

In [ ]:
raw, accepted, unmatched, invalid, provinces, stages = audit(config)
display(raw.head(5))
display(stages)
assert stages.removed.sum() + len(accepted) == len(raw)
display(Markdown(f"**{len(raw):,} observasi mentah → {len(accepted):,} di dalam poligon BIG Kalimantan.**"))

## 2. Mengapa angka dari branch berbeda?

Filter kotak wilayah dapat menerima titik di pulau lain atau menolak bagian Kalimantan jika kotaknya terlalu sempit. Sekarang kami memasangkan **koordinat setiap observasi** ke poligon lima provinsi BIG, sebelum membentuk grid.

Titik yang tidak cocok disimpan untuk audit; tidak otomatis dilabeli error GPS. Alasannya bisa wilayah di luar cakupan, pesisir, perbatasan, atau perbedaan geometri referensi. Titik tepat di batas dua provinsi dihitung sekali dengan label deterministik. Poligon kab/kota Kaka hanya membantu tampilan/detail; tidak menentukan scope utama.

In [ ]:
counts = accepted.groupby("province_name").size().sort_values()
fig, ax = plt.subplots(figsize=(9, 3.5))
counts.plot.barh(ax=ax, color="#517950")
ax.set_xlabel("Jumlah observasi satelit (bukan kejadian unik)")
ax.set_ylabel("")
ax.set_title("Observasi yang masuk poligon lima provinsi Kalimantan")
plt.tight_layout(); plt.show()
display(counts.rename("observations").to_frame())
display(unmatched[["latitude","longitude","acq_date","acq_time"]].head(8))
print("Jumlah belum cocok poligon:", len(unmatched))

## 3. Periode yang benar-benar tersedia

Perbandingan antar-tahun perlu periode sebanding. Hitungan bulanan gabungan dapat berat sebelah karena 2024 dimulai Agustus dan 2026 berakhir Mei. Grafik berikut menampilkan tahun dan bulan terpisah. Hari tanpa observasi dalam CSV tidak membuktikan sensor memperoleh cakupan penuh atau tidak ada api.

In [ ]:
accepted["year"] = pd.to_datetime(accepted.date).dt.year
accepted["month"] = pd.to_datetime(accepted.date).dt.month
coverage = accepted.groupby(["year","month"]).size().unstack().reindex(columns=range(1,13))
display(coverage)
fig, ax = plt.subplots(figsize=(11,3))
im=ax.imshow(coverage, cmap="YlGn", aspect="auto")
ax.set_xticks(range(12), labels=range(1,13)); ax.set_yticks(range(len(coverage)), labels=coverage.index)
ax.set_xlabel("Bulan"); ax.set_ylabel("Tahun")
ax.set_title("Jumlah observasi per tahun-bulan; kosong = tidak ada baris dalam berkas")
fig.colorbar(im, ax=ax, label="Observasi")
plt.tight_layout(); plt.show()
daily = accepted.groupby("date").size()
display(Markdown(f"Rentang **{accepted.date.min()} – {accepted.date.max()} UTC**. Puncak harian **{daily.idxmax()}**, sebanyak **{daily.max():,} observasi**."))

## 4. Confidence, waktu deteksi, dan batas kesimpulan

Confidence membantu membaca kualitas deteksi. Ia tidak membuktikan kebenaran laporan warga dan tidak langsung mengukur urgensi respons. Pengamatan malam hanya menunjukkan waktu deteksi; jenis gambut belum dapat dipastikan.

Selang lintasan satelit tidak mengukur lama verifikasi relawan. Untuk menguji klaim 12–24 jam, dibutuhkan timestamp laporan, penugasan, dan hasil pengecekan lapangan. Dataset ini tidak menyediakan rangkaian tersebut.

In [ ]:
display(accepted.confidence.value_counts().rename("observations").to_frame())
display(accepted.groupby("daynight").size().rename("observations").to_frame())
display(accepted.frp.describe(percentiles=[.5,.9,.95,.99]).to_frame("FRP_MW"))
print("FRP tidak tersedia:", accepted.frp.isna().sum())
print("Koordinat/waktu tidak valid:", len(invalid))
print("Titik tepat di batas bersama:", accepted.border_ambiguous.sum())

## 5. Apa yang digunakan produk?

Hasil audit menyediakan populasi observasi yang jelas untuk peta. Dataset di luar poligon tetap tersimpan sebagai bahan penelusuran, tidak masuk Top 10. Jumlah observasi belum sama dengan jumlah kebakaran; karena itu notebook berikut membentuk area-hari.

Sumber: [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/active_fire/), [BIG Batas Provinsi](https://geoservices.big.go.id/gis/rest/services/STIG/Batas_Provinsi/MapServer). Snapshot dan SHA-256 dicatat ketika notebook 06 dijalankan.